# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
import joblib

## Download Dataset from Kaggle

In [2]:
import kagglehub
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

Using Colab cache for faster access to the 'telco-customer-churn' dataset.


# Load and Clean the Data

In [3]:
data = pd.read_csv(path + '/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
data.shape

(7043, 21)

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
data.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [8]:
data = data.drop('customerID', axis=1)

In [9]:
data.duplicated().sum()

np.int64(22)

In [10]:
data = data.drop_duplicates(keep=False)

In [11]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors='coerce')
data["SeniorCitizen"] = data["SeniorCitizen"].astype('object')

In [12]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7001 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7001 non-null   object 
 1   SeniorCitizen     7001 non-null   object 
 2   Partner           7001 non-null   object 
 3   Dependents        7001 non-null   object 
 4   tenure            7001 non-null   int64  
 5   PhoneService      7001 non-null   object 
 6   MultipleLines     7001 non-null   object 
 7   InternetService   7001 non-null   object 
 8   OnlineSecurity    7001 non-null   object 
 9   OnlineBackup      7001 non-null   object 
 10  DeviceProtection  7001 non-null   object 
 11  TechSupport       7001 non-null   object 
 12  StreamingTV       7001 non-null   object 
 13  StreamingMovies   7001 non-null   object 
 14  Contract          7001 non-null   object 
 15  PaperlessBilling  7001 non-null   object 
 16  PaymentMethod     7001 non-null   object 
 17  

In [13]:
data.isnull().sum()

,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0


In [14]:
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

In [15]:
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [16]:
data['Churn'].value_counts()

,count
Churn,
0,5156
1,1845


# Split Features & Target + Train/Test Split

In [17]:
X = data.drop("Churn", axis=1)
y = data['Churn']

In [18]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Build a preprocessing Pipeline

In [20]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ],

)

# Model Pipeline and Hyperparameter Tuning with GridSearchCV

## Logistic Regression Pipeline + GridSearchCV

In [21]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',LogisticRegression(random_state=42))
])

param_grid_lr = {
    'classifier__C': [0.1, 1, 10],
    'classifier__solver': ['liblinear', 'saga'],
    'classifier__penalty': ['l1', 'l2']
}

grid_search_lr = GridSearchCV(
    lr_pipeline, param_grid_lr, cv=5, n_jobs=-1, scoring='roc_auc', verbose=1
)

grid_search_lr.fit(X_train, y_train)

print("Best LR Parameter: ", grid_search_lr.best_params_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best LR Parameter:  {'classifier__C': 1, 'classifier__penalty': 'l1', 'classifier__solver': 'saga'}


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## Random Forest pipeline + GridSearchCV

In [22]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__class_weight': [None, 'balanced']
}

grid_search_rf = GridSearchCV(
    rf_pipeline, param_grid_rf, cv=5, n_jobs=-1, scoring='roc_auc', verbose=1
)

grid_search_rf.fit(X_train, y_train)

print("Best RF Parameter: ", grid_search_rf.best_params_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best RF Parameter:  {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}


# Evaluate Both Classifiers

In [23]:
best_lr = grid_search_lr.best_estimator_
best_rf = grid_search_rf.best_estimator_

lr_pred_proba = best_lr.predict_proba(X_test)[:, 1]
rf_pred_proba = best_rf.predict_proba(X_test)[:, 1]


In [24]:
print("Logistic Regression Test Perfomance")
print(classification_report(y_test, best_lr.predict(X_test)))
print("LR Accuracy: ", accuracy_score(y_test, best_lr.predict(X_test)))
print("ROC-AUC:", round(roc_auc_score(y_test, lr_pred_proba), 4))

print("Random Forest Test Performance")
print(classification_report(y_test, best_rf.predict(X_test)))
print("RF Accuracy:", accuracy_score(y_test, best_rf.predict(X_test)))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_pred_proba), 4))

Logistic Regression Test Perfomance
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1032
           1       0.64      0.56      0.60       369

    accuracy                           0.80      1401
   macro avg       0.75      0.73      0.73      1401
weighted avg       0.79      0.80      0.80      1401

LR Accuracy:  0.801570306923626
ROC-AUC: 0.8416
Random Forest Test Performance
              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1032
           1       0.54      0.75      0.63       369

    accuracy                           0.77      1401
   macro avg       0.72      0.76      0.73      1401
weighted avg       0.80      0.77      0.78      1401

RF Accuracy: 0.7673090649536045
ROC-AUC: 0.8327


# Export Best Complete Pipeline

In [25]:
if roc_auc_score(y_test, rf_pred_proba) > roc_auc_score(y_test, lr_pred_proba):
    final_pipeline = best_rf
    print("Exporting Random Forest Pipeline")

else:
    final_pipeline = best_lr
    print("Exporting Logistic Regression Pipeline")

joblib.dump(final_pipeline, 'telco_churn_pipeline.joblib')

Exporting Logistic Regression Pipeline


['telco_churn_pipeline.joblib']

# Test Example

In [27]:

loaded_pipeline = joblib.load('telco_churn_pipeline.joblib')

new_df = pd.DataFrame({
    'gender': ['Female'],
    'SeniorCitizen': ['0'],
    'Partner': ['Yes'],
    'Dependents': ['No'],
    'tenure': [7],
    'PhoneService': ['No'],
    'MultipleLines': ['No phone service'],
    'InternetService': ['DSL'],
    'OnlineSecurity': ['No'],
    'OnlineBackup': ['Yes'],
    'DeviceProtection': ['No'],
    'TechSupport': ['No'],
    'StreamingTV': ['No'],
    'StreamingMovies': ['No'],
    'Contract': ['Month-to-month'],
    'PaperlessBilling': ['Yes'],
    'PaymentMethod': ['Electronic check'],
    'MonthlyCharges': [100],
    'TotalCharges': [200]
})
churn_prediction = loaded_pipeline.predict(new_df)
churn_probability = loaded_pipeline.predict_proba(new_df)[:, 1][0]
print(f"Churn Prediction: {churn_prediction}")
print(f"Churn Probability: {churn_probability:.1%}")

Churn Prediction: [0]
Churn Probability: 49.7%


In [30]:

sample = X_test.iloc[:3]
prediction = loaded.predict(sample)
print(f"sample prediction: {prediction}")

sample prediction: [1 0 1]
